# Día 5 · Calibración final del clasificador

Usa las decisiones humanas para reclasificar los elementos extraídos por la v2. Evalúa con leave-one-out para que cada caso se pruebe sin mostrar su propia etiqueta.

In [ ]:
!git clone -q https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git
%cd proyecto_ActividadGrado-Riesgos
!git checkout -q dia-05-extraccion-mejorada
!pip -q install pandas pyarrow openpyxl openai scikit-learn

## 1. Subir dos archivos
Sube simultáneamente `resultados_extraccion_mejorada_dia_05_v2.zip` y `auditoria_final_extraccion_dia_05_v2.xlsx`. La fila incompleta se omitirá automáticamente.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile,shutil
uploaded=files.upload()
zip_name=next(n for n in uploaded if n.startswith('resultados_extraccion_mejorada_dia_05_v2'))
audit_name=next(n for n in uploaded if n.startswith('auditoria_final_extraccion_dia_05_v2'))
source=Path('/content/day5_v2');source.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as z:z.extractall(source)
assert (source/'documentary_items.csv').exists(),'El ZIP no contiene documentary_items.csv'
print('Archivos validados')

## 2. Introducir la clave de OpenAI

In [ ]:
import os
from getpass import getpass
os.environ['OPENAI_API_KEY']=getpass('OPENAI_API_KEY: ')
assert os.environ['OPENAI_API_KEY'].strip(),'La clave está vacía'

## 3. Ejecutar calibración
Primero evaluará los 29 casos con leave-one-out y después clasificará los 1.049 elementos en lotes de diez.

In [ ]:
out=Path('/content/calibracion_dia5');out.mkdir(exist_ok=True)
!python -m src.risk.calibrate_documentary_classifier --items /content/day5_v2/documentary_items.csv --audit "{audit_name}" --chunks data/processed/chunks/chunks_recursive.parquet --output-dir /content/calibracion_dia5 --batch-size 10

## 4. Revisar y descargar

In [ ]:
import json
metrics=json.loads((out/'calibrated_classifier_metrics.json').read_text())
print(json.dumps(metrics,ensure_ascii=False,indent=2))
assert metrics['evaluated']==29,'No se encontraron las 29 etiquetas completas'
assert metrics['items_classified']==1049,'La clasificación no está completa'
zip_path=shutil.make_archive('/content/resultados_clasificador_calibrado_dia_05','zip',out)
files.download(zip_path)